In [1]:
# Importing required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import classification_report

2026-04-27 21:22:17.407743: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777324937.809718      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777324937.924365      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777324938.905082      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777324938.905129      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777324938.905132      23 computation_placer.cc:177] computation placer alr

In [2]:
# Correct the dataset directory path based on your dataset structure
dataset_dir = '/kaggle/input/datasets/shiekhburhan/face-mask-dataset/FMD_DATASET'  # Adjust this if needed

# Create ImageDataGenerator with validation split
img_gen = ImageDataGenerator(rescale=1./255, validation_split=0.15)  # 15% for validation

# Load the training and validation datasets
train_gen = img_gen.flow_from_directory(dataset_dir, 
                                         target_size=(128, 128), 
                                         batch_size=32, 
                                         class_mode='categorical', 
                                         subset='training')  # For training

validation_gen = img_gen.flow_from_directory(dataset_dir, 
                                             target_size=(128, 128), 
                                             b-atch_size=32, 
                                             class_mode='categorical', 
                                             subset='validation')  # For validation

SyntaxError: expression cannot contain assignment, perhaps you meant "=="? (3240236375.py, line 16)

In [ ]:
# Importing necessary library
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Define the path to your dataset
dataset_dir = '/kaggle/input/datasets/shiekhburhan/face-mask-dataset/FMD_DATASET'

# Create ImageDataGenerator for data preprocessing (normalization and validation split)
img_gen = ImageDataGenerator(rescale=1./255, validation_split=0.15)  # Normalizes pixel values by dividing by 255

# Load the training dataset (70% of the total data)
train_gen = img_gen.flow_from_directory(dataset_dir,
                                         target_size=(128, 128),  # Resize images to 128x128
                                         batch_size=32,
                                         class_mode='categorical',
                                         subset='training')  # 70% for training

# Load the validation dataset (15% of the total data)
validation_gen = img_gen.flow_from_directory(dataset_dir,
                                             target_size=(128, 128),  # Resize images to 128x128
                                             batch_size=32,
                                             class_mode='categorical',
                                             subset='validation')  # 15% for validation

# For the test dataset, we can reuse the validation split (15% test data)
test_gen = img_gen.flow_from_directory(dataset_dir,
                                        target_size=(128, 128),  # Resize images to 128x128
                                        batch_size=32,
                                        class_mode='categorical',
                                        subset='validation')  # 15% for testing

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Initialize the CNN model
model = Sequential()

# First convolutional layer + Max-pooling
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)))  # 32 filters, 3x3 kernel
model.add(MaxPooling2D(pool_size=(2, 2)))  # Max-pooling layer with 2x2 pool size

# Second convolutional layer + Max-pooling
model.add(Conv2D(64, (3, 3), activation='relu'))  # 64 filters, 3x3 kernel
model.add(MaxPooling2D(pool_size=(2, 2)))  # Max-pooling layer with 2x2 pool size

# Third convolutional layer + Max-pooling
model.add(Conv2D(128, (3, 3), activation='relu'))  # 128 filters, 3x3 kernel
model.add(MaxPooling2D(pool_size=(2, 2)))  # Max-pooling layer with 2x2 pool size

# Flatten the output of the last convolutional layer
model.add(Flatten())

# Fully connected (dense) layer with dropout regularization
model.add(Dense(128, activation='relu'))  # Fully connected layer with 128 units
model.add(Dropout(0.5))  # Dropout regularization (50%)

# Output layer with softmax activation (for multi-class classification)
model.add(Dense(3, activation='softmax'))  # 3 classes (with_mask, without_mask, incorrect_mask)

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Summary of the model to check the architecture
model.summary()

In [ ]:
# Train the model for 20 epochs
history = model.fit(train_gen,
                    epochs=20,
                    validation_data=validation_gen)

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training and Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Plot training & validation loss values
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training and Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
# First, get the model's predictions on the test set
y_pred = model.predict(test_gen)  # Get the predicted probabilities for each class
y_true = test_gen.classes  # True labels from the test generator

# Convert predicted probabilities to class labels
y_pred_classes = y_pred.argmax(axis=1)  # Convert predictions to class labels

# Compute the confusion matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_true, y_pred_classes)

# Plot confusion matrix as a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=test_gen.class_indices.keys(), yticklabels=test_gen.class_indices.keys())
plt.title('Confusion Matrix')
plt.xlabel('Predicted Labels')
plt.ylabel('True Labels')
plt.show()

# Optionally, you can print the confusion matrix as text
print("Confusion Matrix:")
print(cm)

In [ ]:
from sklearn.metrics import precision_recall_curve
from sklearn.preprocessing import label_binarize

# Binarize the labels for multi-class (One-vs-Rest)
y_true_bin = label_binarize(y_true, classes=[0, 1, 2])  # Adjust for the number of classes
y_pred_bin = y_pred  # These are probabilities for each class

# Compute precision and recall for each class
precision, recall, _ = precision_recall_curve(y_true_bin.ravel(), y_pred_bin.ravel())

# Plot precision-recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, marker='.')
plt.title('Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

# Compute ROC curve for each class (One-vs-Rest)
fpr, tpr, _ = roc_curve(y_true_bin.ravel(), y_pred_bin.ravel())
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', label=f'ROC curve (area = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.title('Receiver Operating Characteristic Curve (ROC Curve)')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc='lower right')
plt.show()

In [ ]:
from sklearn.metrics import classification_report
import pandas as pd

# Generate the classification report (precision, recall, and F1-score)
report = classification_report(y_true, y_pred_classes, target_names=test_gen.class_indices.keys(), output_dict=True)

# Convert the classification report into a pandas DataFrame for better visualization
report_df = pd.DataFrame(report).transpose()

# Display the classification report as a table
print(report_df)

# Optionally, plot the performance metrics for each class in a bar chart
report_df.plot(kind='bar', figsize=(10, 6))  # This will plot precision, recall, F1-score for each class
plt.title('Precision, Recall, F1-Score for Each Class')
plt.ylabel('Score')
plt.xlabel('Classes')
plt.xticks(rotation=0)
plt.show()

In [ ]:
from sklearn.metrics import classification_report

# Print classification report for precision, recall, and F1-score
report = classification_report(y_true, y_pred_classes, target_names=test_gen.class_indices.keys(), output_dict=True)

# Convert the classification report into a dataframe for visualization
import pandas as pd
report_df = pd.DataFrame(report).transpose()

# Plot the performance metrics for each class
report_df.plot(kind='bar', figsize=(10, 6))
plt.title('Precision, Recall, F1-Score for Each Class')
plt.ylabel('Score')
plt.xlabel('Classes')
plt.show()

In [ ]:
# Plot training vs validation accuracy
plt.figure(figsize=(8, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Training vs Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# Analyze the curves
# If training accuracy keeps increasing but validation accuracy plateaus or decreases, it indicates overfitting.
# If both are low and close to each other, it indicates underfitting.

In [ ]:
# Plot training vs validation loss
plt.figure(figsize=(8, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Training vs Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

# Analyze the curves
# If training loss decreases but validation loss increases, this indicates overfitting.
# If both are high and stable, it indicates underfitting.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

# Simple CNN Model with 1 convolutional layer
simple_model = Sequential()
simple_model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)))  # 32 filters, 3x3 kernel
simple_model.add(MaxPooling2D(pool_size=(2, 2)))  # Max-pooling layer
simple_model.add(Flatten())  # Flatten the output
simple_model.add(Dense(3, activation='softmax'))  # Output layer with 3 classes

# Compile the model
simple_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history_simple = simple_model.fit(train_gen, epochs=5, validation_data=validation_gen)

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D

# Load the pre-trained VGG16 model (without the top layers)
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(128, 128, 3))

# Freeze the layers of the base model (we will not train them)
base_model.trainable = False

# Build the complete model with the pre-trained base model and custom output layer
complex_model = Sequential([
    base_model,  # Add the pre-trained VGG16 model
    GlobalAveragePooling2D(),  # Global average pooling to reduce dimensions
    Dense(128, activation='relu'),  # Fully connected layer
    Dropout(0.5),  # Dropout to reduce overfitting
    Dense(3, activation='softmax')  # Output layer with 3 classes
])

# Compile the model
complex_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the model
history_complex = complex_model.fit(train_gen, epochs=5, validation_data=validation_gen)

In [ ]:
# Evaluate the simpler model on the test set
test_acc_simple = simple_model.evaluate(test_gen)

# Evaluate the complex model on the test set
test_acc_complex = complex_model.evaluate(test_gen)

print(f"Simple CNN Test Accuracy: {test_acc_simple[1]}")
print(f"Complex Model (VGG16) Test Accuracy: {test_acc_complex[1]}")

In [ ]:
import time

# Measure time for simple model
start_time = time.time()
simple_model.fit(train_gen, epochs=5, validation_data=validation_gen)
end_time = time.time()
simple_model_training_time = end_time - start_time

# Measure time for complex model
start_time = time.time()
complex_model.fit(train_gen, epochs=5, validation_data=validation_gen)
end_time = time.time()
complex_model_training_time = end_time - start_time

print(f"Simple Model Training Time: {simple_model_training_time} seconds")
print(f"Complex Model Training Time: {complex_model_training_time} seconds")

In [ ]:
# Print model summary for the simple model
simple_model.summary()

# Print model summary for the complex model
complex_model.summary()